In [1]:
%load_ext autoreload
%autoreload 2

from source.dataset import GraphDataset, CombinedDataset, GraphDataModule
from source.model import GravNetModel


In [2]:
import glob
import tqdm
import torch
from torch_geometric.data import DataLoader
import torch.nn as nn 
import torch.nn.functional as F
# from source.train import GravNetLightning
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
import pytorch_lightning as pl
from pytorch_lightning import Trainer
import os
from source.preprocess import preprocess_data

In [ ]:
root_files = glob.glob("/home/benwilson/data/pinpointG4_data/root/10000/*.root")
preprocess_data(root_files, "data/pt/10000/")

In [ ]:
root_files = glob.glob("/home/benwilson/data/pinpointG4_data/root/10001/*.root")
preprocess_data(root_files, "data/pt/10001/")

In [ ]:
root_files = glob.glob("/home/benwilson/data/pinpointG4_data/root/10002/*.root")
preprocess_data(root_files, "data/pt/10002/")

In [ ]:
root_files = glob.glob("/home/benwilson/data/pinpointG4_data/root/10003/*.root")
preprocess_data(root_files, "data/pt/10003/")

In [ ]:
class GravNetLightning(pl.LightningModule):
    def __init__(
        self,
        lr=1e-3,
        lambda_reg=0.5,
    ):
        super().__init__()

        self.save_hyperparameters()

        self.model = GravNetModel()

        self.cls_loss = nn.CrossEntropyLoss()
        self.reg_loss = nn.HuberLoss()

    def forward(self, data):
        return self.model(data)

    def training_step(self, batch, batch_idx):
        class_logits, energy_pred = self(batch)

        cls_loss = self.cls_loss(class_logits, batch.y_class)
        reg_loss = self.reg_loss(energy_pred, batch.y_energy)

        loss = cls_loss + self.hparams.lambda_reg * reg_loss

        preds = torch.argmax(class_logits, dim=1)
        acc = (preds == batch.y_class).float().mean()

        self.log("train_loss", loss, prog_bar=True, batch_size=batch.num_graphs)
        self.log("train_cls_loss", cls_loss, batch_size=batch.num_graphs)
        self.log("train_reg_loss", reg_loss, batch_size=batch.num_graphs)
        self.log("train_acc", acc, prog_bar=True, batch_size=batch.num_graphs)

        return loss

    def validation_step(self, batch, batch_idx):
        class_logits, energy_pred = self(batch)

        cls_loss = self.cls_loss(class_logits, batch.y_class)
        reg_loss = self.reg_loss(energy_pred, batch.y_energy)

        loss = cls_loss + self.hparams.lambda_reg * reg_loss

        preds = torch.argmax(class_logits, dim=1)
        acc = (preds == batch.y_class).float().mean()

        self.log("val_loss", loss, prog_bar=True, batch_size=batch.num_graphs)
        self.log("val_acc", acc, prog_bar=True, batch_size=batch.num_graphs)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.parameters(),
            lr=self.hparams.lr,
        )

        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=50,
        )

        return {
            "optimizer": optimizer,
            "lr_scheduler": scheduler,
        }




In [ ]:
model = GravNetLightning()

# logger = TensorBoardLogger()
        # save_dir=cfg.logging.log_dir,
        # save_dir=hydra.core.hydra_config.HydraConfig.get().runtime.output_dir,    )

pt_files = glob.glob("data/pt/*/*.pt")
assert len(pt_files) > 0, "No .pt files found" 

datamodule = GraphDataModule(
    pt_files=pt_files,
    batch_size=6,
    num_workers=18,
)


checkpoint_cb = ModelCheckpoint(
    monitor="val_loss",
    save_top_k=1,
    mode="min",
)  

early_stop_cb = EarlyStopping(
monitor="val_loss",
min_delta=0,
patience=12,
mode="min",
verbose=True,
)


trainer = Trainer(
    max_epochs=1000,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    callbacks=[checkpoint_cb, early_stop_cb],
    # logger=logger,
    log_every_n_steps=50,
    accumulate_grad_batches=32
)

trainer.fit(model, datamodule=datamodule)
trainer.test(model, datamodule=datamodule)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name     | Type             | Params | Mode  | FLOPs
--------------------------------------------------------------
0 | model    | GravNetModel     | 54.1 K | train | 0    
1 | cls_loss | CrossEntropyLoss | 0      | train | 0    
2 | reg_loss | HuberLoss        | 0      | train | 0    
--------------------------------------------------------------
54.1 K    Trainable params
0         Non-trainable params
54.1 K    Total params
0.217     Total estimated model params size (MB)
53        Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/benwilson/miniforge3/envs/pinpoint/lib/python3.11/site-packages/pytorch_lightning/utilities/data.py:79: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 78167. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.
/home/benwilson/miniforge3/envs/pinpoint/lib/python3.11/site-packages/pytorch_lightning/utilities/data.py:79: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 175373. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.


Training: |          | 0/? [00:00<?, ?it/s]